# Практическая работа №3. Распределенная модель обработки данных

## 0. Назначение практической работы

Цель работы - освоить базовый цикл распределённой обработки табличных данных средствами Apache Spark и PySpark. В данной практике рассматривается как запустить локальное Spark-приложение, подготовить исходные таблицы из базы SQLite, прочитать их в DataFrame, исследовать и задать схему, выполнить преобразования, корректно обработать пропуски и типы, сформировать агрегаты, соединить связанные таблицы, повторить аналитическую задачу через Spark SQL и сохранить итоговый набор данных. Акцент делается не на механическом переносе синтаксиса Pandas в PySpark, а на понимании того, что DataFrame Spark является частью лениво вычисляемого распределённого плана.

Практика выполняется в локальном режиме Spark. Такой режим использует ресурсы одного компьютера и не превращает его в физический кластер, однако сохраняет программную модель Spark, в котором данные разбиваются на разделы, операции над DataFrame формируют план вычислений, а действия запускают выполнение задач. Поэтому один и тот же код после адаптации источников и параметров может быть перенесён на кластерную инфраструктуру.

In [ ]:
print("Hello, World!")


In [ ]:
import pyspark


print(pyspark.__version__)


## 1. Постановка задачи

### 1.1. Источник данных

В качестве исходных данных для выполнения самостоятельной работы используется реляционная база данных SQLite, сохранённая в файле `northwind_small.sqlite`.

Данная база данных представляет собой учебную (сокращённую) версию классической базы данных Northwind, которая эмулирует деятельность компании-оптовика, продающей продукты питания. Согласно указаниям преподавателя, для демонстрации распределённой обработки допускается использование этой облегчённой версии (содержащей 830 заказов и около 2 200 строк в деталях заказов), так как цель работы - освоить программную модель Spark, а не проводить масштабное исследование.

Для обработки данных средствами Apache Spark применяется воспроизводимый staging-этап. Поскольку Spark не имеет встроенного ридера для SQLite, исходные таблицы будут извлечены средствами стандартной библиотеки Python `sqlite3` и модуля `pandas`, а затем сохранены в формате CSV в промежуточный каталог (`spark_input`). Такой подход отделяет слой источника данных от распределённой обработки, избавляет от необходимости подключать внешние JDBC-драйверы и гарантирует воспроизводимость пайплайна.

### 1.2. Предметная область

Предметной областью исследования является анализ оптовой торговли на примере транзакционных данных компании Northwind.

Основные сущности предметной области (из списка таблиц в БД):
*   **Order и OrderDetail:** фактовые таблицы заказов и их позиций (содержат даты оформления, количество товаров, цены и скидки).
*   **Customer:** справочник клиентов (названия компаний, страны, города).
*   **Product и Category:** номенклатура товаров, их категории (напитки, специи, морепродукты и т.д.), а также информация о поставщиках (`Supplier`).
*   **Employee:** данные о сотрудниках (менеджерах, обрабатывающих заказы).

Структура базы данных позволяет строить аналитические витрины уровня транзакций и обогащать их справочной информацией через операции соединения (JOIN).

### 1.3. Цель распределённой обработки

**Главная цель:** Освоить базовый цикл распределённой обработки табличных данных средствами Apache Spark и PySpark на примере базы данных `northwind_small.sqlite`, продемонстрировав перенос программной модели Spark на новую предметную область.

**Конкретные задачи распределённой обработки:**
1. Подготовить источник данных (staging) и инициализировать локальную сессию `SparkSession` (`local[*]`).
2. Прочитать таблицы в Spark DataFrame, исследовать автоматический вывод схемы (`inferSchema`) и применить явную схему `StructType` (например, для таблицы `OrderDetail`, чтобы задать числовые типы для цен, количества и скидок).
3. Выполнить базовые преобразования (`select`, `filter`, `withColumn`), включая создание вычисляемого признака: расчёт чистой выручки по позиции заказа с учётом скидки (формула `UnitPrice * Quantity * (1 - Discount)`).
4. Провести анализ качества данных: выявить пропуски (например, отсутствие региона или факса у некоторых клиентов, а также незаполненные даты отгрузки у заказов) и обоснованно обработать их без потери значимых фактов.
5. Выполнить агрегации (`groupBy().agg()`) для расчёта ключевых бизнес-показателей, таких как суммарные продажи по категориям товаров (`Category`) или по странам клиентов (`Customer.Country`).
6. Объединить (LEFT JOIN / INNER JOIN) фактовые и справочные таблицы (например, `OrderDetail` с `Product` и `Category`) для построения итоговой аналитической витрины и проконтролировать сохранение строк до и после JOIN.
7. Построить зеркальные запросы через Spark SQL (`spark.sql`), используя временные представления (temporary views), и убедиться в эквивалентности результатов с DataFrame API.
8. Сохранить итоговые агрегаты в формате Parquet (и CSV) для последующего чтения и проверить сохранение схемы (round-trip).

**Ограничения анализа:**
Обработка выполняется в локальном режиме (`local[*]`), что использует ресурсы только одного компьютера и не является тестом реальной сетевой масштабируемости. Использование облегчённого набора данных `northwind_small.sqlite` накладывает ограничения на статистическую мощность выводов, однако полностью сохраняет архитектурную логику распределённых вычислений Spark.

## 2. Архитектура входа

### 2.1. Исходный формат

Исходные данные представлены в формате реляционной базы данных SQLite. Формат SQLite хранит таблицы в виде единого бинарного файла, обеспечивая строгую типизацию и поддержку связей, или внешних ключей. Однако Apache Spark не имеет встроенного нативного ридера для файлов SQLite (в отличие от CSV, JSON или Parquet). Прямое чтение потребовало бы подключения внешнего JDBC-драйвера, что усложнило бы воспроизводимость кода.

### 2.2. Промежуточный этап - Staging

Для бесшовной интеграции SQLite и Spark применяется **staging-этап**.
С помощью стандартного Python-модуля `sqlite3` и библиотеки `pandas` данные извлекаются из базы и сохраняются в промежуточный каталог `spark_input` в виде CSV-файлов.
Этот подход решает несколько задач:
* Отделяет слой извлечения данных (Extract) от распределённой обработки (Transform).
* Позволяет Spark читать данные нативно, используя встроенный CSV-ридер.
* Обеспечивает прозрачность: любой студент или преподаватель может проверить исходные CSV-файлы.
* Исключает проблемы совместимости версий JDBC-драйверов.

**Извлекаемые таблицы (основные сущности):**
* `Order` - заголовки заказов.
* `OrderDetail` - построчные позиции заказов (содержит цены и скидки).
* `Product` - справочник товаров.
* `Category` - категории товаров.
* `Customer` - справочник клиентов.

### 2.3. Схема связанных DataFrame

После чтения CSV-файлов в Spark формируется набор логически связанных DataFrame, отражающих структуру реляционной базы данных:

1. **df_orders (Order):**
   * Ключ: `Id`
   * Содержит: `CustomerId`, `EmployeeId`, `OrderDate`, `ShippedDate`.
2. **df_order_details (OrderDetail):**
   * Ключ: `OrderId`, `ProductId` (составной)
   * Содержит: `UnitPrice`, `Quantity`, `Discount` (используются для расчетов).
3. **df_products (Product):**
   * Ключ: `Id`
   * Содержит: `ProductName`, `CategoryId`, `SupplierId`.
4. **df_categories (Category):**
   * Ключ: `Id`
   * Содержит: `CategoryName`.
5. **df_customers (Customer):**
   * Ключ: `Id`
   * Содержит: `CompanyName`, `Country`, `City`.

**Логика связей для будущих JOIN:**
* `df_order_details` соединяется с `df_products` по ключу `ProductId`.
* `df_products` соединяется с `df_categories` по ключу `CategoryId`.
* `df_orders` соединяется с `df_order_details` по ключу `OrderId`.
* `df_orders` соединяется с `df_customers` по ключу `CustomerId`.

Такая схема типа «звезда» (факт `OrderDetail` + измерения) позволяет выполнять агрегацию выручки по категориям товаров и странам клиентов.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path


# 1. Create Path constants
DB_PATH = Path("northwind_small.sqlite")

STAGING = Path("spark_input")
STAGING.mkdir(exist_ok=True)

# 2. List of tables that need to be extracted
TABLES = ["Order", "OrderDetail", "Product", "Category", "Customer"]

# 3. Extract from SQLite to CSV
with sqlite3.connect(DB_PATH) as conn:
    for table in TABLES:
        # Read one table
        df_pd = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)

        # Create CSV file based on current table
        df_pd.to_csv(STAGING / f"{table}.csv", index=False, encoding="utf-8")

        # Print CSV file info
        print(f"{table:15} -> {len(df_pd):>5} rows saved to {table}.csv")


### 2.4. Выводы

Набор данных представляет собой реляционную модель интернет-магазина Northwind, извлечённую из SQLite в staging-каталог в виде набора связанных CSV-файлов общим объёмом около 3 200 строк (главные факты - 830 заказов и 2 155 позиций). Данные **пригодны для распределённой обработки**, а схема типа «звезда» (фактовая таблица `OrderDetail` с измерениями `Product`, `Category`, `Customer`) позволяет отвечать на ключевые бизнес-вопросы о структуре продаж и географии клиентов; детальное применение явных схем, трансформаций и JOIN-операций будет выполнено в следующих разделах.

## 3. SparkSession и схема

### 3.1. Версии

В данном разделе инициализируется точка входа для работы с распределёнными данными, фиксируются версии окружения, а также исследуются методы работы со схемами DataFrame: автоматическое определение типов (inferSchema) и явное задание контракта (StructType).

In [ ]:
from pyspark.sql import SparkSession
import sys


# 1. Inittialization of SparkSession
spark = (
    SparkSession.builder
        .appName("Northwind_Small_Practice")
        .master("local[*]")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.shuffle.partitions", "8")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# 2. Pinning versions
print("=" * 50)
print("ИНФОРМАЦИЯ ОБ ОКРУЖЕНИИ")
print("=" * 50)
print(f"Python version: {sys.version.split()[0]}")
print(f"Java version:   {spark.sparkContext._jvm.java.lang.System.getProperty('java.version')}")
print(f"Spark version:  {spark.version}")
print(f"Master:         {spark.sparkContext.master}")
print(f"App Name:       {spark.sparkContext.appName}")
print("=" * 50)


При создании `SparkSession` мы явно указали параметр `master("local[*]")`. Важно понимать, что в отличие от промышленного кластера, где Driver (управляющий процесс) и Executors (вычислительные процессы) физически распределены по разным серверам, в режиме `local[*]` все они эмулируются внутри одной JVM на нашем локальном компьютере. Символ `*` означает, что Spark попытается задействовать все доступные логические ядра процессора. Это позволяет нам использовать программную модель Spark (разделение на партиции, ленивые вычисления) без необходимости развертывать реальную сетевую инфраструктуру.

Параметр `spark.driver.memory=2g` ограничивает память, доступную для драйвера. В локальном режиме драйвер и экзекуторы делят оперативную память хоста, поэтому важно задавать разумные лимиты, чтобы избежать ошибок `OutOfMemory` при выполнении тяжелых операций (например, `collect()` или `JOIN` больших таблиц).

### 3.2. Сравнение inferSchema и StructType

In [ ]:
from pathlib import Path
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType


STAGING = Path("spark_input")

# --- Вариант 1: Автоматический вывод схемы (inferSchema) ---
print("1. Автоматическая схема (inferSchema=True):")

df_details_inferred = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(STAGING / "OrderDetail.csv"))
)

df_details_inferred.printSchema()
print(f"Строк: {df_details_inferred.count()}\n")

# --- Вариант 2: Явная схема (StructType) ---
print("2. Явная схема (StructType):")

order_details_schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("OrderId", IntegerType(), True),
    StructField("ProductId", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True)
])

df_details_explicit = (
    spark.read
        .option("header", True)
        .schema(order_details_schema)
        .csv(str(STAGING / "OrderDetail.csv"))
)

df_details_explicit.printSchema()
print(f"Строк: {df_details_explicit.count()}")


**Автоматическая схема (inferSchema=True):**
Spark совершает дополнительный проход по файлу, чтобы определить типы. В нашем случае он корректно определил числовые типы (`int` для Id, `double` для UnitPrice и Discount), однако этот подход требует больше времени при чтении и может ошибаться на грязных данных (например, если в колонке с числами встретится строка, Spark переведёт весь столбец в `string`).

**Явная схема (StructType):**
Мы сами задали контракт данных. Это ускоряет чтение (Spark не тратит время на анализ), гарантирует строгую типизацию и служит "документацией" для других разработчиков. В Big Data практике явная схема (Explicit Schema) является предпочтительной для продакшн-пайплайнов.

### 3.3. Анализ преобразования временных меток

In [ ]:
from pyspark.sql import functions as F


# 1. Читаем Order (по умолчанию все поля string)
df_orders_raw = (
    spark.read
        .option("header", True)
        .csv(str(STAGING / "Order.csv"))
)

# 2. Преобразуем строковые даты в TimestampType
date_cols = ["OrderDate", "RequiredDate", "ShippedDate"]
df_orders = df_orders_raw

for c in date_cols:
    df_orders = df_orders.withColumn(c, F.to_timestamp(F.col(c), "yyyy-MM-dd"))
    # Примечание: в SQLite даты могут быть в формате "yyyy-MM-dd HH:mm:ss" или просто "yyyy-MM-dd"
    # Если формат другой, строку формата нужно адаптировать.

print("Схема Order после преобразования дат:")
df_orders.printSchema()

print("\nПример данных (OrderDate, ShippedDate):")
df_orders.select("Id", "CustomerId", "OrderDate", "ShippedDate").show(5, truncate=False)


При чтении CSV-файлов Spark по умолчанию не знает, что столбцы `OrderDate` и `ShippedDate` являются датами - для него это просто текст (`StringType`). Это создает скрытую угрозу при анализе: если мы попытаемся сравнить даты или найти разницу между ними, Spark применит *лексикографическое* (алфавитное) сравнение. В формате `yyyy-MM-dd` это может случайно сработать, но как только появится время (часы и минуты) или изменится формат (например, `dd-MM-yyyy`), все вычисления будут давать логически неверные результаты без выдачи явной ошибки (Silent bug).

Использование `F.to_timestamp` переводит строки в строгий тип `TimestampType`. С этого момента Spark понимает внутреннюю структуру времени, позволяя нам использовать функции вроде `datediff()` (разница в днях) или `date_format()` (извлечение месяца и года). Важно отметить, что в Spark DataFrame операции `withColumn` являются иммутабельными (immutable): мы не изменяем исходный `df_orders_raw`, а создаем *новый* DataFrame `df_orders`, который содержит новый план вычислений с преобразованным столбцом. Само преобразование физически произойдет только при вызове Action (например, `show()` или `count()`), что наглядно демонстрирует принцип ленивых вычислений (Lazy Evaluation).

### 3.4. Выводы

В ходе выполнения раздела была успешно инициализирована локальная сессия `SparkSession` с мастер-узлом `local[*]`, что подтверждает готовность среды PySpark к выполнению распределённых трансформаций. Установлено, что DataFrame Spark концептуально отличается от Pandas: он лениво вычисляется, а его схема может быть как выведена автоматически (`inferSchema`), так и задана строго через `StructType`. Для таблиц с числовыми метриками (например, `OrderDetail`) явное задание схемы является более надёжным и быстрым подходом. Строковые поля с временными метками (таблица `Order`) были приведены к типу `TimestampType`, что в дальнейшем позволит применять к ним функции работы со временем (`datediff`, `date_format`), а не полагаться на случайное лексикографическое сравнение строк.

## 4. Преобразования данных - Transformations

В данном разделе применяются базовые трансформации Spark DataFrame (`select`, `filter`, `withColumn`).

Главный акцент сделан на создании **вычисляемого признака (derived feature)** - фактической выручки по позиции заказа, которая рассчитывается на основе цены, количества и процента скидки. Также мы наглядно убедимся в принципе **Lazy Evaluation**: трансформации описывают план, а реальное вычисление и вывод числа строк происходит только при вызове Action (`count()`).

### 4.1. Создание вычисляемого признака выручки

В таблице `OrderDetail` скидка (`Discount`) представлена в виде десятичной дроби (например, 0.05 означает 5%). Используя метод `withColumn`, мы создадим новый столбец `LineRevenue`, который рассчитывает чистую выручку от продажи конкретной позиции с учетом скидки.

In [ ]:
# 1. Создаем вычисляемый признак "LineRevenue"
df_details_enriched = df_details_explicit.withColumn(
    "LineRevenue", 
    F.round(F.col("UnitPrice") * F.col("Quantity") * (1 - F.col("Discount")), 2)
)

print("Схема после добавления вычисляемого признака:")
df_details_enriched.printSchema()


### 4.2. Выбор необходимых столбцов и фильтрация

С помощью `select()` мы оставим только те поля, которые понадобятся для дальнейшего анализа. Затем применим `filter()`, чтобы отобрать только крупные оптовые заказы (где количество `Quantity >= 50`), по которым была предоставлена скидка (`Discount > 0`). Затем мы вызовем **Action** `count()`, чтобы узнать, сколько строк прошло фильтр.

In [ ]:
# 2. Выбираем нужные столбцы (Transformation: select)
df_details_selected = df_details_enriched.select(
    "OrderId", "ProductId", "Quantity", "Discount", "LineRevenue"
)

# 3. Применяем фильтрацию (Transformation: filter)
high_volume_discounts = df_details_selected.filter(
    (F.col("Quantity") >= 50) & (F.col("Discount") > 0)
)

# 4. Вызов Action для подсчета строк
count_rows = high_volume_discounts.count()

print(f"Всего позиций в исходной таблице: {df_details_selected.count()}")
print(f"Позиций с Quantity >= 50 и Discount > 0: {count_rows}")


### 4.3. Сортировка отфильтрованных данных

Чтобы визуально оценить результаты фильтрации, отсортируем полученный DataFrame по убыванию вычисленного признака выручки (`LineRevenue`) с помощью `orderBy()` и выведем первые 5 строк методом `show()`.

In [ ]:
# 5. Вывод результата (Action: show)
print("Топ-5 самых дорогих оптовых позиций (с учетом скидки):")
high_volume_discounts.orderBy(F.desc("LineRevenue")).show(5, truncate=False)


### 4.4. Поиск уникальных заказов

С помощью трансформации `distinct()` найдем, сколько уникальных заказов (`OrderId`) содержат хотя бы одну позицию, прошедшую наш фильтр. Это покажет, в скольких конкретных сделках встречались крупные оптовые покупки со скидкой.

In [ ]:
# 6. Удаление дубликатов (Transformation: distinct)
unique_orders = high_volume_discounts.select("OrderId").distinct()

print(f"Уникальных заказов с крупными оптовыми позициями: {unique_orders.count()}")

# Вывод первых 10 уникальных ID
unique_orders.show(10, truncate=False)


### 4.5. Интерпретация результатов трансформаций

В ходе данного раздела мы применили полный цикл трансформаций:
1. **`withColumn`**: создали столбец `LineRevenue`. Это яркий пример аналитического правила, которое описывается для всего столбца сразу (векторно), без использования циклов Python.
2. **`select`**: сократили размерность данных, оставив только необходимые для анализа столбцы.
3. **`filter`**: отсекли строки, не представляющие интереса для текущей задачи.
4. **`orderBy`**: упорядочили данные для просмотра самых крупных сделок.
5. **`distinct`**: получили уникальные ID заказов.

Обратите внимание на принцип **Lazy Evaluation**. Когда мы вызывали `withColumn` или `filter`, в памяти не происходило никаких вычислений - Spark лишь строил логический план. Только при вызове `count()` и `show()` (которые являются **Action**) оптимизатор Spark запустил выполнение задач (Tasks) на партициях.

### 4.6. Выводы

Применение распределённых трансформаций позволило обогатить исходный набор данных новым вычисляемым признаком `LineRevenue` (фактическая выручка с учётом скидки) и выделить сегмент крупных оптовых заказов. Строгое разделение на ленивые трансформации и триггерные действия (Actions) гарантирует, что Spark оптимизирует весь конвейер обработки (DataFrame Plan) перед фактическим обращением к памяти. Количество строк после фильтрации подтверждает, что данные успешно прошли проверку на соответствие бизнес-условиям.

## 5. Качество данных

В данном разделе проводится диагностика пропущенных значений (null). Наличие `null` в Spark не всегда означает ошибку: часто это структурная особенность данных (например, заказ еще не доставлен, поэтому дата доставки отсутствует). Мы исследуем распределение пропусков, чтобы принять обоснованное решение: заполнить их (fillna), отфильтровать (filter) или оставить как есть.

### 5.1. Подсчёт пропусков в таблице Order

Проверим наличие пропусков в ключевых временных полях заказов. Если дата отгрузки (`ShippedDate`) пуста, это может означать, что заказ еще не отправлен.

In [ ]:
# Считаем пропуски для каждого поля
null_counts = df_orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in df_orders.columns
])

print("Количество пропусков (null) по столбцам Order:")
null_counts.show(vertical=True)


### 5.2. Предметная проверка пропусков - Анализ по группам

Удалять строки с пропущенной датой отгрузки - грубая ошибка. Сначала выясним, с чем связаны эти пропуски. Для этого отфильтруем строки, где `ShippedDate` is null, и сгруппируем их по странам клиентов (через JOIN с `Customer`), чтобы понять, это локальная проблема логистики или общая.

In [ ]:
# Читаем клиентов (пока только нужные колонки)
df_customers = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "Customer.csv"))

# Соединяем Order и Customer
df_orders_customers = df_orders.join(
    df_customers.select("Id", "Country", "CompanyName"), 
    df_orders["CustomerId"] == df_customers["Id"], 
    "left"
)

# Ищем заказы без даты отгрузки и группируем по странам
missing_shipped = (
    df_orders_customers
        .filter(F.col("ShippedDate").isNull())
        .groupBy("Country")
        .count()
        .orderBy(F.desc("count"))
)

print("Страны, в которых есть заказы без даты отгрузки (топ-5):")
missing_shipped.show(5, truncate=False)


### 5.3. Заполнение пропусков категориального признака

В таблице `Customer` поле `Region` (регион клиента) часто бывает пустым. В отличие от дат, мы не можем просто проигнорировать регион при анализе географии. Заполним эти пропуски строкой `"Unknown"`, создав отдельную аналитическую категорию, чтобы не потерять клиента из выборки.

In [ ]:
# Проверяем, сколько пустых регионов
missing_region_count = df_customers.filter(F.col("Region").isNull()).count()
print(f"Клиентов без указания региона: {missing_region_count}")

# Заполняем null значением "Unknown"
df_customers_clean = df_customers.fillna({"Region": "Unknown"})

# Проверяем результат
print("\nРезультат после fillna (проверяем, что null больше нет):")
df_customers_clean.filter(F.col("Region") == "Unknown").show(3, truncate=False)


Все три формата **успешно прошли round-trip**: число строк и столбцов совпадает с оригиналом (12 384 × 6). Однако **dtypes различаются**.

**Parquet сохраняет схему полностью**, тогда как **CSV теряет типы** - категориальные признаки превращаются в `object`, а `int16` - в `int64`. Это означает, что после чтения CSV потребуется **повторная оптимизация dtypes**.

Для аналитического pipeline **Parquet предпочтительнее**: он сохраняет типы, занимает в 3.6 раз меньше места и поддерживает выборочное чтение столбцов. **CSV** остаётся удобным для обмена и просмотра в текстовом редакторе. **JSON** - оптимален для полуструктурированного взаимодействия, но избыточен по размеру.

### 5.4. Интерпретация результатов и обоснование действий

1. **Пропуски в датах (`ShippedDate`):** Анализ показал, что пропуски распределены по разным странам. Это означает, что отсутствие даты - это не ошибка сбора данных, а **бизнес-состояние** (заказ находится в обработке или отменен). Мы **не будем** удалять эти строки (dropna) и **не будем** заполнять их текущей датой, так как это исказит время доставки. При расчете логистики мы просто отфильтруем только те заказы, где `ShippedDate` не null.

2. **Пропуски в категориях (`Region`):** Регион - это справочный признак. Если мы удалим клиентов без региона, мы потеряем часть выручки. Заполнение значением `"Unknown"` (fillna) позволяет сохранить транзакции в общем объеме и отдельно оценить долю клиентов с неуточненной географией.

### 5.5. Выводы

Анализ качества данных показал, что пропуски в наборе Northwind носят структурный характер. Присутствие `null` в датах отгрузки отражает незавершённые бизнес-процессы, поэтому удаление таких строк (dropna) было бы методологически неверным. Для категориальных признаков (например, `Region`) применено обоснованное заполнение значением `"Unknown"` (fillna), что позволило сохранить целостность фактовых данных для дальнейшей агрегации без создания фиктивных числовых величин. Преобразование типов на предыдущих этапах обеспечило корректную работу с датами и числами.

## 6. Агрегации и JOIN

В данном разделе мы объединим фактовые таблицы (заказы, детали заказов) со справочниками (товары, категории) и выполним агрегацию для получения бизнес-показателей. Особое внимание уделяется контролю количества строк до и после JOIN, чтобы избежать нежелательного размножения данных (many-to-many).

### 6.1. Агрегация фактов - несколько агрегатных функций

Сначала сгруппируем данные в таблице `OrderDetail` по `ProductId` и рассчитаем три метрики одновременно: общее количество проданных единиц (`SUM(Quantity)`), среднюю цену (`AVG(UnitPrice)`) и общую выручку (`SUM(LineRevenue)` из нашего вычисляемого признака). Это покажет топ-5 самых продаваемых товаров.

In [ ]:
product_sales_stats = (
    df_details_enriched
        .groupBy("ProductId")
        .agg(
            F.sum("Quantity").alias("total_quantity_sold"),
            F.round(F.avg("UnitPrice"), 2).alias("avg_unit_price"),
            F.round(F.sum("LineRevenue"), 2).alias("total_revenue")
        )
        .orderBy(F.desc("total_revenue"))
)

print("Топ-5 товаров по выручке (ProductId):")
product_sales_stats.show(5, truncate=False)


### 6.2. INNER JOIN и контроль количества строк

Чтобы узнать названия категорий (вместо сухих `ProductId`), выполним `INNER JOIN` с таблицами `Product` и `Category`.
Перед JOIN важно запомнить исходное число строк (оно равно количеству уникальных товаров в агрегате). После JOIN оно не должно измениться, так как у каждого товара ровно одна категория.

In [ ]:
# Читаем справочники
df_products = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "Product.csv"))
df_categories = spark.read.option("header", True).option("inferSchema", True).csv(str(STAGING / "Category.csv"))

# Считаем строки ДО JOIN
rows_before = product_sales_stats.count()
print(f"Строк до JOIN: {rows_before}")

# Выполняем цепочку INNER JOIN (OrderDetail stats -> Product -> Category)
product_sales_with_categories = (
    product_sales_stats
        .join(df_products.select("Id", "ProductName", "CategoryId"), 
            product_sales_stats["ProductId"] == df_products["Id"], 
            "inner")
        .join(df_categories.select("Id", "CategoryName"), 
            df_products["CategoryId"] == df_categories["Id"], 
            "inner")
        .select(
            df_categories["CategoryName"],
            df_products["ProductName"],
            "total_quantity_sold",
            "avg_unit_price",
            "total_revenue"
        )
        .orderBy(F.desc("total_revenue"))
)

# Считаем строки ПОСЛЕ JOIN
rows_after = product_sales_with_categories.count()
print(f"Строк после JOIN: {rows_after}")

print("\nТоп-5 самых прибыльных товаров с названиями категорий:")
product_sales_with_categories.show(5, truncate=False)


### 6.3. LEFT JOIN и географический анализ

Выполним `LEFT JOIN` между таблицей заказов (`Order`) и клиентами (`Customer`). `LEFT JOIN` гарантирует, что мы не потеряем заказы, даже если по какой-то причине у заказа нет совпадения в таблице клиентов (хотя в идеальной БД такого быть не должно). После объединения сгруппируем заказы по странам (`Country`), чтобы увидеть, откуда приходит больше всего заказов.

In [ ]:
# LEFT JOIN
orders_with_customers = df_orders.join(
    df_customers_clean.select("Id", "Country", "City"), 
    df_orders["CustomerId"] == df_customers_clean["Id"], 
    "left"
)

# Агрегация по странам (2-я группировка)
orders_by_country = (
    orders_with_customers
        .groupBy("Country")
        .agg(
            F.count(df_orders["Id"]).alias("total_orders"),
            F.countDistinct("CustomerId").alias("unique_customers")
        )
        .orderBy(F.desc("total_orders"))
)

print("Топ-5 стран по количеству заказов (LEFT JOIN):")
orders_by_country.show(5, truncate=False)


### 6.4. Интерпретация результатов JOIN и агрегаций

**Контроль строк:** При выполнении `INNER JOIN` между агрегированными продажами (`product_sales_stats`), товарами (`Product`) и категориями (`Category`) количество строк до и после объединения **совпало**. Это доказывает, что связи "один-ко-многим" были соблюдены корректно, и мы не получили декартово произведение (дубликаты).

**Бизнес-выводы:** 
1. Агрегация по товарам позволила выявить самые прибыльные позиции. Благодаря вычисляемому признаку `LineRevenue`, мы видим чистую выручку с учетом скидок.
2. `LEFT JOIN` с клиентами и последующая группировка по `Country` показали географическую структуру продаж. В Northwind (как и в реальном бизнесе) лидируют крупные рынки (например, USA и Germany). Использование `LEFT JOIN` здесь безопасно: заказы без страны не потеряются, а попадут в категорию `null`, что позволит нам позже оценить качество данных.

### 6.5. Выводы

Раздел продемонстрировал силу Spark при объединении распределённых данных. Мы успешно применили множественные агрегатные функции (`sum`, `avg`, `count`) в рамках одного `groupBy().agg()`, что эффективнее, чем выполнять их по отдельности. Операции `INNER JOIN` и `LEFT JOIN` позволили обогатить фактовые данные справочной информацией (названия товаров, категории, страны клиентов). Строгий контроль количества строк до и после JOIN подтвердил отсутствие аномалий в связях таблиц. Полученные агрегаты (топ-товары и география продаж) готовы к сохранению в формате Parquet для построения BI-дашбордов.

## 7. Spark SQL

Spark SQL предоставляет декларативный интерфейс для работы с данными, позволяя писать привычные SQL-запросы. Важно понимать, что Spark SQL и DataFrame API - это не два разных движка, а два способа описать один и тот же план вычислений (Logical Plan). В этом разделе мы зарегистрируем наши DataFrame как временные представления (Temporary Views) и выполним аналитические запросы.

### 7.1. Создание временных представлений - Temporary Views

Чтобы использовать SQL, сначала нужно зарегистрировать DataFrame в текущей сессии `SparkSession` с помощью метода `createOrReplaceTempView()`. Это создаст "виртуальную таблицу", которая существует только во время работы текущей сессии.

In [ ]:
# --- Регистрируем подготовленные таблицы для SQL ---
# 1. Таблица фактов с выручкой (наша витрина из JOIN)
product_sales_with_categories.createOrReplaceTempView("sales_view")

# 2. Исходная таблица заказов
df_orders.createOrReplaceTempView("orders_view")

# 3. Таблица клиентов
df_customers_clean.createOrReplaceTempView("customers_view")

print("Временные представления успешно созданы.")


### 7.2. SQL-запрос: Выручка по категориям (GROUP BY + ORDER BY)

Напишем SQL-запрос, который агрегирует общую выручку по каждой категории товаров и сортирует их по убыванию. Этот запрос использует `GROUP BY` и `ORDER BY` по вычисленному агрегату, что соответствует требованиям задания.

In [ ]:
sql_query_1 = """
SELECT 
    CategoryName,
    COUNT(ProductName) AS unique_products,
    SUM(total_quantity_sold) AS total_items_sold,
    ROUND(SUM(total_revenue), 2) AS total_revenue
FROM sales_view
GROUP BY CategoryName
ORDER BY total_revenue DESC
"""

category_revenue_sql = spark.sql(sql_query_1)
print("Топ категорий по выручке (через Spark SQL):")
category_revenue_sql.show(truncate=False)


### 7.3. SQL-запрос: География заказов и сопоставление с DataFrame API

Второй запрос определит количество заказов по странам. Затем мы выполним **точно такую же логику** через DataFrame API (методы `groupBy` и `agg`), чтобы доказать, что оба подхода дают идентичный результат.

In [ ]:
# --- 1. Через Spark SQL ---
sql_query_2 = """
SELECT 
    Country,
    COUNT(*) AS total_orders
FROM orders_view o
LEFT JOIN customers_view c ON o.CustomerId = c.Id
GROUP BY Country
ORDER BY total_orders DESC
"""
orders_by_country_sql = spark.sql(sql_query_2)

print("Результат через Spark SQL (Топ-5):")
orders_by_country_sql.show(5, truncate=False)


# --- 2. Через DataFrame API (Сравнение) ---
orders_by_country_df = (
    df_orders.alias("o")
        .join(df_customers_clean.alias("c"), F.col("o.CustomerId") == F.col("c.Id"), "left")
        .groupBy("Country")
        .agg(F.count("*").alias("total_orders"))
        .orderBy(F.desc("total_orders"))
)

print("Результат через DataFrame API (Топ-5):")
orders_by_country_df.show(5, truncate=False)

# Проверка на полное совпадение
are_equal = orders_by_country_sql.collect() == orders_by_country_df.collect()
print(f"\nСовпадают ли результаты SQL и DataFrame API? -> {are_equal}")


### 7.4. Интерпретация результатов Spark SQL

1. **Декларативность:** Запросы Spark SQL позволяют описать *что* мы хотим получить (например, сумму выручки по категориям), не задумываясь о том, *как* именно Spark будет перемешивать данные (shuffle). Оптимизатор Catalyst сам решает, как лучше выполнить `JOIN` и `GROUP BY`.
2. **Эквивалентность API:** Результат прямого сравнения списков (`collect()`) показал значение `True`. Это доказывает, что Spark SQL (строка запроса) и DataFrame API (цепочка методов) компилируются в один и тот же физический план. Выбор между ними - это вопрос удобства: аналитикам проще писать SQL, а инженерам данных - использовать DataFrame API для интеграции в код приложения.
3. **Бизнес-ценность:** SQL-запросы подтвердили, что категории с наибольшей выручкой совпадают с теми, что мы видели в DataFrame API. Это говорит о корректности наших предыдущих вычислений.

### 7.5. Выводы

Использование временных представлений (`createOrReplaceTempView`) позволило применить стандартный SQL для анализа распределённых данных. Мы убедились, что Spark SQL предоставляет мощный декларативный инструмент, который полностью интегрирован с DataFrame API. Сложные запросы с `GROUP BY` и сортировкой агрегатов выполняются оптимизатором Catalyst так же эффективно, как и вызовы методов PySpark. Такая гибкость делает Spark универсальной платформой, подходящей как для дата-инженеров (пишущих код), так и для аналитиков (владеющих SQL).

## 8. Экспорт

Spark записывает DataFrame не как один файл, а как **каталог**, внутри которого лежат несколько part-файлов (по одному на каждый раздел/партицию). Parquet - это колоночный формат, который сохраняет схему и идеально подходит для аналитики. CSV удобен для обмена, но не хранит типы данных.

### 8.1. Сохранение витрины в Parquet и CSV

Сначала сохраним итоговую аналитическую витрину (с агрегатами по товарам и категориям) в формат Parquet для дальнейшей обработки. Для удобства проверки дополнительно сохраним её в CSV. 
Поскольку витрина содержит небольшое количество строк, для CSV мы используем `coalesce(1)`, чтобы получить один аккуратный файл вместо множества мелких кусочков.

In [ ]:
# Определяем путь для сохранения
OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)

# 1. Сохранение в Parquet (режим overwrite позволяет перезаписывать результат)
(
    product_sales_with_categories
        .write
        .mode("overwrite")
        .parquet(str(OUT / "sales_parquet"))  # <-- Will fix soon
)
print("Данные успешно сохранены в Parquet.")

# 2. Сохранение в CSV (объединяем в 1 файл с заголовком)
(
    product_sales_with_categories
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(str(OUT / "sales_csv"))
)
print("Данные успешно сохранены в CSV.")
